# Deteksi Hoaks dengan Transformer (IndoBERT)

Notebook ini menyiapkan klasifikasi biner **hoaks (1)** vs **berita benar (0)** menggunakan model IndoBERT. Dataset Komdigi yang tersedia hanya berisi artikel klarifikasi hoaks, jadi diperlukan CSV berita benar yang terverifikasi untuk melatih kelas negatif. Jalankan sel berurutan dan atur `DATASET_BENAR_PATH` di bagian konfigurasi.

> Untuk mengurangi kebocoran label, model hanya memakai judul artikel dan membuang awalan `[HOAKS]`. Hasil model tetap perlu diuji pada data yang mewakili klaim nyata; model ini bukan pengganti verifikasi fakta.

## 1. Instalasi dependensi
Jalankan sekali bila paket belum tersedia. Model pretrained diunduh dari Hugging Face saat sel training dijalankan.

In [4]:
%pip install -q transformers datasets accelerate scikit-learn torch

Note: you may need to restart the kernel to use updated packages.


## 2. Konfigurasi dan impor

In [5]:
from pathlib import Path
import random
import re

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 128
ROOT_DIR = Path.cwd()
while ROOT_DIR.name.lower() in ["transformer", "05_transformer", "notebooks"]:
    ROOT_DIR = ROOT_DIR.parent

OUTPUT_DIR = (ROOT_DIR / "notebooks" / "05_transformer" / "model_indo_bert_hoaks") if (ROOT_DIR / "notebooks" / "05_transformer").exists() else Path("model_indo_bert_hoaks")

KOMDIGI_PATH = ROOT_DIR / "data" / "komdigi_hoaks.csv"
UPLOADS_DIR = ROOT_DIR / "web" / "uploads"
preferred_valid_path = UPLOADS_DIR / "berita_valid_terbaru.csv"
valid_candidates = sorted(
    UPLOADS_DIR.glob("berita_valid*.csv"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
) if UPLOADS_DIR.is_dir() else []
DATASET_BENAR_PATH = (
    preferred_valid_path
    if preferred_valid_path.is_file()
    else (valid_candidates[0] if valid_candidates else preferred_valid_path)
)

print("Perangkat:", "CUDA" if torch.cuda.is_available() else "CPU")
print("Root project:", ROOT_DIR.resolve())
print("Dataset Komdigi:", KOMDIGI_PATH.resolve())
print("Dataset berita valid:", DATASET_BENAR_PATH.resolve())

Perangkat: CPU
Root project: D:\Kuliah\Kelas\Semester 5\Product Based Learning\HoaxCheckML
Dataset Komdigi: D:\Kuliah\Kelas\Semester 5\Product Based Learning\HoaxCheckML\data\komdigi_hoaks.csv
Dataset berita valid: D:\Kuliah\Kelas\Semester 5\Product Based Learning\HoaxCheckML\web\uploads\berita_valid_terbaru.csv


## 3. Memuat dan memeriksa data
CSV berita valid berada di `web/uploads` (misalnya `berita_valid_terbaru.csv`). Sel konfigurasi otomatis memilih file `berita_valid*.csv` terbaru jika berkas `terbaru` tidak ditemukan.

In [6]:
if not KOMDIGI_PATH.is_file():
    raise FileNotFoundError(f"Dataset Komdigi tidak ditemukan: {KOMDIGI_PATH.resolve()}")
if not DATASET_BENAR_PATH.is_file():
    available = list(UPLOADS_DIR.glob("*.csv")) if UPLOADS_DIR.is_dir() else []
    available_names = ", ".join(path.name for path in available) or "tidak ada file CSV"
    raise FileNotFoundError(
        f"Dataset berita valid tidak ditemukan di {UPLOADS_DIR.resolve()}. "
        f"File CSV di folder tersebut: {available_names}. "
        "Pastikan berita_valid_terbaru.csv tersedia di folder web/uploads/ lalu jalankan ulang sel."
    )

hoaks_raw = pd.read_csv(KOMDIGI_PATH, low_memory=False, encoding="utf-8-sig")
benar_raw = pd.read_csv(
    DATASET_BENAR_PATH,
    sep=None,
    engine="python",
    encoding="utf-8-sig",
)

def normalisasi_nama_kolom(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")

def pilih_kolom_teks(frame: pd.DataFrame, aliases: tuple[str, ...]) -> str:
    normalized = {normalisasi_nama_kolom(column): column for column in frame.columns}
    for alias in aliases:
        if alias in normalized:
            return normalized[alias]
    raise ValueError(
        "Kolom teks tidak ditemukan. Kolom yang tersedia: "
        f"{list(frame.columns)}. Nama yang didukung: {list(aliases)}."
    )

kolom_hoaks = pilih_kolom_teks(
    hoaks_raw, ("title", "judul", "headline", "judul_berita")
)
kolom_benar = pilih_kolom_teks(
    benar_raw,
    ("title", "judul", "headline", "judul_berita", "text", "teks", "content", "konten", "isi", "body_text", "berita"),
)
hoaks = hoaks_raw[[kolom_hoaks]].rename(columns={kolom_hoaks: "text"}).copy()
benar = benar_raw[[kolom_benar]].rename(columns={kolom_benar: "text"}).copy()

hoaks["label"] = 1
benar["label"] = 0
data = pd.concat([hoaks, benar], ignore_index=True)
data["text"] = data["text"].fillna("").astype(str).str.strip()
data["text"] = data["text"].str.replace(r"^\s*\[\s*hoaks\s*\]\s*", "", regex=True, case=False)
data = data[data["text"].str.len() > 0].copy()

# Hapus teks identik lintas kelas agar duplikat tidak bocor ke train dan test.
data["normalized_text"] = data["text"].str.casefold().str.replace(r"\s+", " ", regex=True)
label_counts_per_text = data.groupby("normalized_text")["label"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1].index
data = data[~data["normalized_text"].isin(conflicting_texts)]
data = data.drop_duplicates(subset="normalized_text").drop(columns="normalized_text").reset_index(drop=True)

counts = data["label"].value_counts().reindex([0, 1], fill_value=0)
print(f"Jumlah berita benar (0): {counts[0]:,}")
print(f"Jumlah hoaks (1): {counts[1]:,}")
print(f"Teks duplikat dengan label bertentangan yang dihapus: {len(conflicting_texts):,}")
if (counts < 10).any():
    raise ValueError("Setiap kelas membutuhkan sedikitnya 10 contoh unik agar split train/validasi/test dapat dilakukan.")
data.sample(min(5, len(data)), random_state=SEED)

Jumlah berita benar (0): 749
Jumlah hoaks (1): 4,162
Teks duplikat dengan label bertentangan yang dihapus: 0


,text,label
915,Wakil Presiden Gibran Siap Damaikan Iran denga...,1
3227,Warga Satu Kampung Mendadak Kaya karena Judi O...,1
3290,Surat Penipuan Mengatasnamakan Direktur Peruma...,1
4316,Letda Prajurit Kopassus Jadi Tersangka Kelalai...,0
4208,"43,95 Juta Perjalanan KA Jarak Jauh dan Lokal,...",0


## 4. Membagi data
Pembagian dibuat berstrata agar proporsi kelas terjaga: 70% train, 15% validasi, dan 15% test. Test set hanya dipakai untuk evaluasi akhir.

In [7]:
train_val, test_df = train_test_split(
    data, test_size=0.15, random_state=SEED, stratify=data["label"]
)
train_df, val_df = train_test_split(
    train_val,
    test_size=0.15 / 0.85,
    random_state=SEED,
    stratify=train_val["label"],
)
print(f"Train: {len(train_df):,} | Validasi: {len(val_df):,} | Test: {len(test_df):,}")
print("Distribusi label train:")
print(train_df["label"].value_counts().sort_index())

Train: 3,437 | Validasi: 737 | Test: 737
Distribusi label train:
label
0     525
1    2912
Name: count, dtype: int64


## 5. Tokenisasi dan fine-tuning IndoBERT

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "BERITA_BENAR", 1: "HOAKS"},
    label2id={"BERITA_BENAR": 0, "HOAKS": 1},
)

def tokenisasi(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

def ke_dataset(frame: pd.DataFrame) -> Dataset:
    dataset = Dataset.from_pandas(
        frame[["text", "label"]].reset_index(drop=True), preserve_index=False
    )
    return dataset.map(tokenisasi, batched=True, remove_columns=["text"])

train_ds = ke_dataset(train_df)
val_ds = ke_dataset(val_df)
test_ds = ke_dataset(test_df)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def hitung_metrik(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=hitung_metrik,
)
trainer.train()

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

c:\Users\qulbiazzumi\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\qulbiazzumi\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.
c:\Users\qulbiazzumi\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\qulbiazzumi\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


... (13 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\qulbiazzumi\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1290, training_loss=0.10051336916842202, metrics={'train_runtime': 1506.9368, 'train_samples_per_second': 6.842, 'train_steps_per_second': 0.856, 'total_flos': 90194675332920.0, 'train_loss': 0.10051336916842202, 'epoch': 3.0})

## 6. Evaluasi akhir dan penyimpanan model
Perhatikan precision/recall/F1 untuk kelas hoaks, bukan hanya akurasi. Jika data berita benar tidak seimbang atau berbeda sumber, metrik dapat mencerminkan bias dataset.

In [9]:
test_output = trainer.predict(test_ds)
y_true = test_df["label"].to_numpy()
y_pred = np.argmax(test_output.predictions, axis=-1)
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1],
    target_names=["Berita benar", "Hoaks"],
    zero_division=0,
))
print("Confusion matrix (baris=aktual, kolom=prediksi):")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Model dan tokenizer tersimpan di: {OUTPUT_DIR.resolve()}")

c:\Users\qulbiazzumi\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

Berita benar       0.92      0.92      0.92       112
       Hoaks       0.99      0.99      0.99       625

    accuracy                           0.98       737
   macro avg       0.95      0.95      0.95       737
weighted avg       0.98      0.98      0.98       737

Confusion matrix (baris=aktual, kolom=prediksi):
[[103   9]
 [  9 616]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model dan tokenizer tersimpan di: D:\Kuliah\Kelas\Semester 5\Product Based Learning\HoaxCheckML\transformer\model_indo_bert_hoaks


## 7. Mencoba prediksi teks baru
Masukkan judul atau klaim singkat. Prediksi hanya indikasi berdasarkan data latih, bukan keputusan faktual.

In [10]:
def prediksi_hoaks(teks: str) -> dict:
    if not isinstance(teks, str) or not teks.strip():
        raise ValueError("Teks input tidak boleh kosong.")
    model.eval()
    inputs = tokenizer(
        teks.strip(),
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)
    with torch.no_grad():
        probabilities = torch.softmax(model(**inputs).logits, dim=-1)[0]
    predicted_label = int(torch.argmax(probabilities).item())
    return {
        "prediksi": "HOAKS" if predicted_label == 1 else "BERITA_BENAR",
        "probabilitas_hoaks": round(float(probabilities[1].item()), 4),
        "catatan": "Prediksi model bukan pengganti pemeriksaan fakta.",
    }

prediksi_hoaks("Masukkan judul berita atau klaim yang ingin diperiksa di sini")

{'prediksi': 'BERITA_BENAR',
 'probabilitas_hoaks': 0.0048,
 'catatan': 'Prediksi model bukan pengganti pemeriksaan fakta.'}